In [ ]:
!pip install transformers pykeen torch tqdm requests pandas  

In [ ]:
# This installs the core GNN library and the necessary dependencies
!pip install -q torch-geometric transformers pykeen tqdm requests pandas

In [ ]:
import torch


pt_version = torch.__version__.split('+')[0]
cuda_version = "cu" + torch.version.cuda.replace(".", "")


url = f"https://data.pyg.org/whl/torch-{pt_version}+{cuda_version}.html"
print(f"Fetching PyG libraries for Torch {pt_version} and {cuda_version}...")
print(f"Download URL: {url}")


!pip install -q pyg-lib torch-sparse torch-scatter -f {url}



## Text alignment with node ID's

In [ ]:
import torch
import requests
import os
import pandas as pd
from pykeen.datasets import FB15k237


dataset = FB15k237()
entity_to_id = dataset.training.entity_to_id
num_nodes = len(entity_to_id)
print(f" Total Nodes in Graph: {num_nodes}")

# Create the robust dictionary
robust_entity_to_id = {}
for pykeen_id, int_id in entity_to_id.items():
    standardized_id = str(pykeen_id).strip('/').replace('/', '.')
    robust_entity_to_id[standardized_id] = int_id
    robust_entity_to_id[pykeen_id] = int_id

print("\nDownloading Wikipedia Text Descriptions from KG-BERT repo")
# THE FIX: The correct author of KG-BERT is yao8839836. 
text_url = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2text.txt"
text_file = "entity2text.txt"

# Force a redownload by deleting the old broken "404" file if it exists
if os.path.exists(text_file):
    os.remove(text_file)

# Download and explicitly check for 404 errors
response = requests.get(text_url)
response.raise_for_status() 

with open(text_file, 'w', encoding='utf-8') as f:
    f.write(response.text)


df_text = pd.read_csv(text_file, sep='\t', header=None, names=['freebase_id', 'description'])


node_texts = [""] * num_nodes
match_count = 0

for _, row in df_text.iterrows():
    fb_id = str(row['freebase_id'])
    desc = str(row['description'])
    
    standardized_fb_id = fb_id.strip('/').replace('/', '.')
    
    if standardized_fb_id in robust_entity_to_id:
        node_id = robust_entity_to_id[standardized_fb_id]
        
        if node_texts[node_id] == "":
            node_texts[node_id] = desc
            match_count += 1

print(f"matched {match_count} out of {num_nodes} node descriptions!")

# Print out a sample to prove it actually worked!
sample_id = list(robust_entity_to_id.values())[0]
print(f"\nExample Text for Node {sample_id}:")
print(f"\"{node_texts[sample_id][:150]}...\"")

## Learning text embeddings

In [ ]:

import torch
from transformers import DistilBertTokenizer, DistilBertModel
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using DistilBERT on: {device} for learning embeddings")

# 1. Load Pre-trained Model and Tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased').to(device)
model.eval() # We are extracting, not training, so eval() is mandatory

# 2. Storage for our final vectors
embedding_dim = 768
semantic_embeddings = torch.zeros((num_nodes, embedding_dim))

# 3. Extraction Loop (Batched for GPU Safety)
batch_size = 128
print(f"\n Extracting {embedding_dim}-D semantic vectors for {num_nodes} entities")

with torch.no_grad():
    for i in tqdm(range(0, num_nodes, batch_size), desc="Processing Text"):
        batch_texts = node_texts[i : i + batch_size]
        
        # Tokenize the text (truncate to 128 words to save memory/speed)
        inputs = tokenizer(batch_texts, return_tensors='pt', padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Forward pass through DistilBERT
        outputs = model(**inputs)
        
        # Grab the [CLS] token (the first token at index 0) which holds the sentence meaning
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        
        # Move back to CPU to avoid Kaggle crashes and store in our master tensor
        semantic_embeddings[i : i + batch_size] = cls_embeddings.cpu()

# 4. Save the Master Embedding Matrix
save_path = "fb15k237_distilbert_embeddings.pt"
torch.save(semantic_embeddings, save_path)

print("\n" + "="*60)
print(f" Semantic Matrix saved: {semantic_embeddings.shape}")
print("="*60)

In [ ]:
#Attention gateway
import torch
import torch.nn as nn

class SemanticGateway(nn.Module):
    def __init__(self, num_nodes, text_embedding_path, gnn_dim=128):
        super(SemanticGateway, self).__init__()
        
        # 1. Load the 768-D Master Matrix we just created
        raw_text_embs = torch.load(text_embedding_path)
        
        # 2. Freeze the text embeddings so we don't destroy DistilBERT's knowledge
        # We store them in an nn.Embedding, but tell PyTorch NOT to update them
        self.static_text_embs = nn.Embedding.from_pretrained(raw_text_embs, freeze=True)
        
        # 3. The Bridge Layer: Compress 768 -> 128
        # This layer IS learnable. It figures out what text features matter for the graph.
        distilbert_dim = 768
        self.projection_layer = nn.Sequential(
            nn.Linear(distilbert_dim, 256),
            nn.ReLU(),  # Add a little non-linearity to catch complex text patterns
            nn.Dropout(0.2), # Prevent overfitting
            nn.Linear(256, gnn_dim)
        )

    def forward(self, node_ids):
        # Step A: Get the raw 768-D text vector
        raw_text = self.static_text_embs(node_ids)
        
        # Step B: Compress it to 128-D using our learnable bridge
        compressed_text = self.projection_layer(raw_text)
        
        return compressed_text


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gateway = SemanticGateway(num_nodes=14505, text_embedding_path='fb15k237_distilbert_embeddings.pt', gnn_dim=128).to(device)


test_nodes = torch.tensor([0, 1, 2]).to(device)
output_vectors = gateway(test_nodes)

print("\n" + "="*50)
print(f"Input Nodes:{test_nodes.shape}")
print(f"Output Vectors:{output_vectors.shape}")
print("="*50)

## 1)Using SARMP on R-Gat

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGATConv

class SemanticRGAT(torch.nn.Module):
    # Changed to 2 heads to fit inside Kaggle's 15GB VRAM
    def __init__(self, num_nodes, num_relations, hidden_channels, text_embedding_path, heads=2):
        super().__init__()
        
        self.gateway = SemanticGateway(num_nodes, text_embedding_path, gnn_dim=hidden_channels)
        self.rel_emb = nn.Embedding(num_relations, hidden_channels)

        out_channels = hidden_channels // heads
        
        
        self.conv1 = RGATConv(in_channels=hidden_channels, out_channels=out_channels, 
                              num_relations=num_relations, heads=heads, concat=True,
                              num_bases=30)
        self.norm1 = nn.LayerNorm(hidden_channels)

        self.conv2 = RGATConv(in_channels=hidden_channels, out_channels=out_channels, 
                              num_relations=num_relations, heads=heads, concat=True,
                              num_bases=30)
        self.norm2 = nn.LayerNorm(hidden_channels)

        self.drop = nn.Dropout(0.3)

    def encode(self, edge_index, edge_type):
        all_node_ids = torch.arange(self.gateway.static_text_embs.num_embeddings, device=edge_index.device)
        x_init = self.gateway(all_node_ids)
        
        # Layer 1 with Residuals
        x_updated = self.conv1(x_init, edge_index, edge_type)
        x = self.norm1(x_updated) + x_init
        x = self.drop(F.relu(x))
        
        # Layer 2 with Residuals
        x_updated = self.conv2(x, edge_index, edge_type)
        x = self.norm2(x_updated) + x
        
        return x, self.rel_emb.weight

    def decode(self, head_emb, rel_id, node_embs, rel_embs, custom_tails=None):
        r_emb = rel_embs[rel_id]
        if custom_tails is None:
            return (head_emb * r_emb) @ node_embs.t()
        else:
            tail_emb = node_embs[custom_tails]
            return ((head_emb * r_emb) * tail_emb).sum(dim=-1)

## 2)Using SARMP on CompGCN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing


class CompConv(MessagePassing):
    def __init__(self, in_channels, out_channels, op="mult"):
        super().__init__(aggr="mean")
        self.op = op
        self.W_node = nn.Linear(in_channels, out_channels, bias=False)
        self.W_rel = nn.Linear(in_channels, out_channels, bias=False)

    def forward(self, x, edge_index, edge_type, rel_emb):
        edge_features = rel_emb[edge_type]
        x_updated = self.propagate(edge_index, x=x, edge_features=edge_features)
        x_updated = self.W_node(x_updated)
        rel_emb_updated = self.W_rel(rel_emb)
        return x_updated, rel_emb_updated

    def message(self, x_j, edge_features):
        if self.op == "mult": return x_j * edge_features
        elif self.op == "sub": return x_j - edge_features
        else: raise ValueError("Operation must be 'mult' or 'sub'")


class SemanticCompGCN(torch.nn.Module):
    def __init__(self, num_nodes, num_relations, hidden_channels, text_embedding_path, op="mult"):
        super().__init__()
        
        #Distillbert embeddings instead of random ones
        self.gateway = SemanticGateway(num_nodes, text_embedding_path, gnn_dim=hidden_channels)
        self.rel_emb = nn.Embedding(num_relations, hidden_channels)

        # 2. Layers, Norms, and Dropout
        self.conv1 = CompConv(in_channels=hidden_channels, out_channels=hidden_channels, op=op)
        self.norm1 = nn.LayerNorm(hidden_channels)

        self.conv2 = CompConv(in_channels=hidden_channels, out_channels=hidden_channels, op=op)
        self.norm2 = nn.LayerNorm(hidden_channels)

        self.drop = nn.Dropout(0.3)

    def encode(self, edge_index, edge_type):
        
        all_node_ids = torch.arange(self.gateway.static_text_embs.num_embeddings, device=edge_index.device)
        x_init = self.gateway(all_node_ids)
        edge_attr_init = self.rel_emb.weight

        # Layer 1 with Residuals
        x_updated, edge_attr_updated = self.conv1(x_init, edge_index, edge_type, edge_attr_init)
        x = self.norm1(x_updated) + x_init
        x = self.drop(F.relu(x))
        edge_attr = edge_attr_updated

        # Layer 2 with Residuals
        x_updated, edge_attr_updated = self.conv2(x, edge_index, edge_type, edge_attr)
        x = self.norm2(x_updated) + x
        edge_attr = edge_attr_updated

        return x, edge_attr

    
    # def decode(self, head_emb, rel_id, node_embs, rel_embs, custom_tails=None):
    #     r_emb = rel_embs[rel_id]
        
    #     # --- THE FIX ---
    #     # If validation passes a single relation, force it to be 2D (1, 128)
    #     if r_emb.dim() == 1:
    #         r_emb = r_emb.unsqueeze(0)
    #     # ---------------
            
    #     if custom_tails is None:
    #         # Validation/Testing Phase: Score against ALL 14,505 tails
    #         # head_emb: (Batch, 128), r_emb: (Batch, 128), node_embs: (14505, 128)
    #         distance = torch.norm(head_emb.unsqueeze(1) + r_emb.unsqueeze(1) - node_embs.unsqueeze(0), p=1, dim=-1)
    #         # Return negative distance because MarginLoss wants the "True" score to be a higher number!
    #         return -distance 
    #     else:
    #         # Training Phase: Score against specific mini-batch tails
    #         tail_emb = node_embs[custom_tails]
    #         distance = torch.norm(head_emb + r_emb - tail_emb, p=1, dim=-1)
    #         return -distance
            
    def decode(self, head_emb, rel_id, node_embs, rel_embs, custom_tails=None):
        r_emb = rel_embs[rel_id]
        if custom_tails is None:
            return (head_emb * r_emb) @ node_embs.t()
        else:
            tail_emb = node_embs[custom_tails]
            return ((head_emb * r_emb) * tail_emb).sum(dim=-1)





## Data loader for this model 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from pykeen.datasets import FB15k237
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
text_path = 'fb15k237_distilbert_embeddings.pt'

pk_dataset = FB15k237()

class GraphData: pass
data = GraphData()
data.num_nodes = pk_dataset.training.num_entities
data.edge_index = torch.stack([pk_dataset.training.mapped_triples[:, 0], pk_dataset.training.mapped_triples[:, 2]], dim=0).to(device)
data.edge_type = pk_dataset.training.mapped_triples[:, 1].to(device)

val_data = GraphData()
val_data.edge_index = torch.stack([pk_dataset.validation.mapped_triples[:, 0], pk_dataset.validation.mapped_triples[:, 2]], dim=0).to(device)
val_data.edge_type = pk_dataset.validation.mapped_triples[:, 1].to(device)

num_nodes = data.num_nodes
num_rels = int(data.edge_type.max()) + 1

def check_validation_mrr(model, train_graph, val_graph):
    model.eval()
    num_val_edges = val_graph.edge_index.size(1)
    mrr_sum = 0.0
    with torch.no_grad():
        node_emb, rel_emb = model.encode(train_graph.edge_index, train_graph.edge_type)
        eval_indices = torch.randperm(num_val_edges)[:1000]
        for i in eval_indices:
            h = val_graph.edge_index[0, i]
            r = val_graph.edge_type[i]
            true_t = val_graph.edge_index[1, i]
            scores = model.decode(node_emb[h].unsqueeze(0), r, node_emb, rel_emb)
            sorted_indices = torch.argsort(scores, descending=True).squeeze()
            rank = (sorted_indices == true_t).nonzero(as_tuple=True)[0].item() + 1
            mrr_sum += 1.0 / rank
    return mrr_sum / len(eval_indices)

## Training

### 1) custom semantic compgcn 

In [ ]:
HIDDEN_DIM_COMP = 128
NUM_NEGATIVES = 10
BATCH_SIZE = 50000
EPOCHS = 1000
PATIENCE = 20 # 20 checks = 500 epochs of patience

print("\nStarting Training: Semantic_CompGCN")
model_comp = SemanticCompGCN(num_nodes, num_rels, HIDDEN_DIM_COMP, text_path, op="sub").to(device)
optimizer = torch.optim.AdamW(model_comp.parameters(), lr=0.001, weight_decay=1e-4)

edge_indices = torch.arange(data.edge_index.size(1), device=device)
dataloader = DataLoader(edge_indices, batch_size=BATCH_SIZE, shuffle=True)

best_val_mrr = 0.0
patience_counter = 0
best_weights = None

for epoch in range(1, EPOCHS + 1):
    model_comp.train()
    total_loss = 0
    
    for batch_idx in dataloader:
        batch_idx = batch_idx.to(device)
        optimizer.zero_grad()
        
        node_embs, rel_embs = model_comp.encode(data.edge_index, data.edge_type)
        
        heads = data.edge_index[0, batch_idx]
        rels = data.edge_type[batch_idx]
        true_tails = data.edge_index[1, batch_idx]
        
        neg_heads = heads.repeat(NUM_NEGATIVES)
        neg_rels = rels.repeat(NUM_NEGATIVES)
        fake_tails = torch.randint(0, num_nodes, (heads.size(0) * NUM_NEGATIVES,), device=device)
        
        pos_scores = model_comp.decode(node_embs[heads], rels, node_embs, rel_embs, custom_tails=true_tails)
        neg_scores = model_comp.decode(node_embs[neg_heads], neg_rels, node_embs, rel_embs, custom_tails=fake_tails)
        
        pos_scores_repeated = pos_scores.repeat(NUM_NEGATIVES)
        target = torch.ones_like(neg_scores)
        loss = F.margin_ranking_loss(pos_scores_repeated, neg_scores, target, margin=1.0)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 25 == 0:
        avg_loss = total_loss / len(dataloader)
        val_mrr = check_validation_mrr(model_comp, data, val_data)
        print(f"[Semantic_CompGCN] Epoch {epoch:04d} | Loss: {avg_loss:.4f} | Val MRR: {val_mrr:.4f}")
        
        if val_mrr > best_val_mrr:
            best_val_mrr = val_mrr
            best_weights = copy.deepcopy(model_comp.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            print(f"Stopped early at epoch {epoch}. Best MRR: {best_val_mrr:.4f}")
            break
            
if best_weights is not None:
    model_comp.load_state_dict(best_weights)
torch.save(model_comp.state_dict(), "Semantic_CompGCN_best.pt")

### 2) semantic R-gat

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import LinkNeighborLoader
import copy


HIDDEN_DIM_RGAT = 64 
BATCH_SIZE = 2048  # Number of true edges per batch
NUM_NEGATIVES = 10 
EPOCHS = 1000
PATIENCE = 15

print("\n Starting Training: Semantic_RGAT (Subgraph Sampled)")

# Initializing r-gat model
model_rgat = SemanticRGAT(num_nodes, num_rels, HIDDEN_DIM_RGAT, text_path, heads=2).to(device)
optimizer = torch.optim.AdamW(model_rgat.parameters(), lr=0.001, weight_decay=1e-4)

# Subgraph sampling
train_loader = LinkNeighborLoader(
    data=data,
    num_neighbors=[10, 10], # Keep a max of 10 neighbors per node, for 2 layers deep
    edge_label_index=data.edge_index,
    edge_label=torch.ones(data.edge_index.size(1)), # True edges get a label of 1
    neg_sampling_ratio=NUM_NEGATIVES, # Automatically creates 10 fake edges per real one!
    batch_size=BATCH_SIZE,
    shuffle=True,
)

best_val_mrr = 0.0
patience_counter = 0
best_weights = None


for epoch in range(1, EPOCHS + 1):
    model_rgat.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        
        node_embs, rel_embs = model_rgat.encode(batch.edge_index, batch.edge_type)
        
        
        heads = batch.edge_label_index[0]
        tails = batch.edge_label_index[1]
        relations = batch.edge_type[:len(heads)] # Match relations to the sampled edges
        
        
        # edge_label is 1.0 for real edges, 0.0 for fake edges
        is_true_edge = (batch.edge_label == 1.0)
        is_fake_edge = (batch.edge_label == 0.0)
        
        
        # We use a 1-to-1 dot product for the sampled edges
        h_emb = node_embs[heads]
        r_emb = rel_embs[relations]
        t_emb = node_embs[tails]
        scores = ((h_emb * r_emb) * t_emb).sum(dim=-1)
        
        # Separate Positives and Negatives
        pos_scores = scores[is_true_edge]
        neg_scores = scores[is_fake_edge]
        
        # Margin Ranking Loss
        # Because we sampled dynamically, we align the shapes for the loss function
        # (This averages the negatives against the positives for the batch)
        if len(pos_scores) > 0 and len(neg_scores) > 0:
            # Re-shape for margin loss: repeat positives to match negatives
            pos_scores_repeated = pos_scores.repeat_interleave(NUM_NEGATIVES)[:len(neg_scores)]
            target = torch.ones_like(neg_scores)
            
            loss = F.margin_ranking_loss(pos_scores_repeated, neg_scores, target, margin=1.0)
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    if epoch % 10 == 0:
        avg_loss = total_loss / len(train_loader)
        val_mrr = check_validation_mrr(model_rgat, data, val_data)
        print(f"[Semantic_RGAT] Epoch {epoch:04d} | Loss: {avg_loss:.4f} | Val MRR: {val_mrr:.4f}")
        
        if val_mrr > best_val_mrr:
            best_val_mrr = val_mrr
            best_weights = copy.deepcopy(model_rgat.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            print(f"Stopped early at epoch {epoch}. Best MRR: {best_val_mrr:.4f}")
            break

if best_weights is not None:
    model_rgat.load_state_dict(best_weights)
torch.save(model_rgat.state_dict(), "Semantic_RGAT_best.pt")

### Semantic-R-GCN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.nn import RGCNConv
import copy


class SemanticRGCN(torch.nn.Module):
    def __init__(self, num_nodes, num_relations, hidden_channels, text_embedding_path, num_bases=30):
        super().__init__()
        
        # 768 -> 128 Dimension text compressor
        self.gateway = SemanticGateway(num_nodes, text_embedding_path, gnn_dim=hidden_channels)
        self.rel_emb = nn.Embedding(num_relations, hidden_channels)

        # RGCN Layers using your preferred num_bases=30
        self.conv1 = RGCNConv(hidden_channels, hidden_channels, num_relations, num_bases=num_bases)
        self.norm1 = nn.LayerNorm(hidden_channels)

        self.conv2 = RGCNConv(hidden_channels, hidden_channels, num_relations, num_bases=num_bases)
        self.norm2 = nn.LayerNorm(hidden_channels)

        self.drop = nn.Dropout(0.3)

    def encode(self, edge_index, edge_type):
        all_node_ids = torch.arange(self.gateway.static_text_embs.num_embeddings, device=edge_index.device)
        x_init = self.gateway(all_node_ids)
        
        # Layer 1
        x = self.conv1(x_init, edge_index, edge_type)
        x = self.norm1(x)
        x = self.drop(F.relu(x))
        
        # Layer 2
        x = self.conv2(x, edge_index, edge_type)
        x = self.norm2(x)
        
        return x, self.rel_emb.weight

    # We use DistMult here so it is fairly compared against your CompGCN
    # Swap this into your SemanticRGCN class!
    # def decode(self, head_emb, rel_id, node_embs, rel_embs, custom_tails=None):
    #     r_emb = rel_embs[rel_id]
        
    #     # Broadcasting fix for validation step
    #     if r_emb.dim() == 1:
    #         r_emb = r_emb.unsqueeze(0)
            
    #     if custom_tails is None:
    #         # Score against ALL tails
    #         distance = torch.norm(head_emb.unsqueeze(1) + r_emb.unsqueeze(1) - node_embs.unsqueeze(0), p=1, dim=-1)
    #         return -distance
    #     else:
    #         # Score against specific fake tails
    #         tail_emb = node_embs[custom_tails]
    #         distance = torch.norm(head_emb + r_emb - tail_emb, p=1, dim=-1)
    #         return -distance
            
    def decode(self, head_emb, rel_id, node_embs, rel_embs, custom_tails=None):
        r_emb = rel_embs[rel_id]
        if custom_tails is None:
            return (head_emb * r_emb) @ node_embs.t()
        else:
            tail_emb = node_embs[custom_tails]
            return ((head_emb * r_emb) * tail_emb).sum(dim=-1)


#Hyperparameters
HIDDEN_DIM = 128
NUM_BASES = 30
NUM_NEGATIVES = 10
BATCH_SIZE = 25000  
EPOCHS = 800
PATIENCE = 4

print(f"\n Semantic-RGCN: {HIDDEN_DIM} Dims, {NUM_BASES} Bases, {NUM_NEGATIVES} Negatives.")
model_rgcn = SemanticRGCN(num_nodes, num_rels, HIDDEN_DIM, text_path, num_bases=NUM_BASES).to(device)


optimizer = torch.optim.Adam(model_rgcn.parameters(), lr=0.005, weight_decay=1e-5)

edge_indices = torch.arange(data.edge_index.size(1), device=device)
dataloader = DataLoader(edge_indices, batch_size=BATCH_SIZE, shuffle=True)

best_val_mrr = 0.0
patience_counter = 0
best_weights = None


for epoch in range(1, EPOCHS + 1):
    model_rgcn.train()
    total_loss = 0
    
    for batch_idx in dataloader:
        batch_idx = batch_idx.to(device)
        optimizer.zero_grad()
        
        # 1. Full-Batch Encode
        node_embs, rel_embs = model_rgcn.encode(data.edge_index, data.edge_type)
        
        # 2. Extract Mini-Batch Edges
        heads = data.edge_index[0, batch_idx]
        rels = data.edge_type[batch_idx]
        true_tails = data.edge_index[1, batch_idx]
        
        # 3. Generate Negatives
        neg_heads = heads.repeat(NUM_NEGATIVES)
        neg_rels = rels.repeat(NUM_NEGATIVES)
        fake_tails = torch.randint(0, num_nodes, (heads.size(0) * NUM_NEGATIVES,), device=device)
        
        # 4. Decode and Score
        pos_scores = model_rgcn.decode(node_embs[heads], rels, node_embs, rel_embs, custom_tails=true_tails)
        neg_scores = model_rgcn.decode(node_embs[neg_heads], neg_rels, node_embs, rel_embs, custom_tails=fake_tails)
        
        # 5. Margin Ranking Loss
        pos_scores_repeated = pos_scores.repeat(NUM_NEGATIVES)
        target = torch.ones_like(neg_scores)
        loss = F.margin_ranking_loss(pos_scores_repeated, neg_scores, target, margin=1.0)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation 
    if epoch % 25 == 0:
        avg_loss = total_loss / len(dataloader)
        print(f"[Semantic_RGCN] Epoch {epoch:04d} | Loss: {avg_loss:.4f} | Running Validation...")
        
        val_mrr = check_validation_mrr(model_rgcn, data, val_data)
        print(f"          -> Validation MRR: {val_mrr:.4f}")
        
        if val_mrr > best_val_mrr:
            best_val_mrr = val_mrr
            best_weights = copy.deepcopy(model_rgcn.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"          -> No improvement. Patience: {patience_counter}/{PATIENCE}")
            
        if patience_counter >= PATIENCE:
            print(f"\n Early Stopping Triggered! Best MRR: {best_val_mrr:.4f}")
            break
            

if best_weights is not None:
    model_rgcn.load_state_dict(best_weights)
torch.save(model_rgcn.state_dict(), "Semantic_RGCN_best_Distmult.pt")

## Inference on all Semantic models

### Trans-E Decoder models

In [ ]:

import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pykeen.datasets import FB15k237

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pk_dataset = FB15k237()

# Get datasets
train_triples = pk_dataset.training.mapped_triples.numpy()
test_triples = pk_dataset.testing.mapped_triples.numpy()

# --- 1. TOPOLOGY ANALYZER ---
print("Analyzing Graph Topology from Training Data")
df_train = pd.DataFrame(train_triples, columns=['head', 'relation', 'tail'])
df_test = pd.DataFrame(test_triples, columns=['head', 'relation', 'tail'])

# Node Degrees
node_degrees = df_train['head'].value_counts().add(df_train['tail'].value_counts(), fill_value=0)
hub_nodes = set(node_degrees[node_degrees >= 100].index)
rare_nodes = set(node_degrees[node_degrees <= 10].index)

# Relation Types
rel_stats = df_train.groupby('relation').agg(
    heads_per_tail=('head', lambda x: len(x) / len(x.unique())),
    tails_per_head=('tail', lambda x: len(x) / len(x.unique()))
)
rel_types = {}
for rel, row in rel_stats.iterrows():
    hpt, tph = row['heads_per_tail'], row['tails_per_head']
    if hpt < 1.5 and tph < 1.5: rel_types[rel] = '1-to-1'
    elif hpt < 1.5 and tph >= 1.5: rel_types[rel] = '1-to-N'
    elif hpt >= 1.5 and tph < 1.5: rel_types[rel] = 'N-to-1'
    else: rel_types[rel] = 'N-to-N'

df_test['rel_category'] = df_test['relation'].map(rel_types)
df_test['is_hub'] = df_test['tail'].isin(hub_nodes)
df_test['is_rare'] = df_test['tail'].isin(rare_nodes)

# --- 2. THE EVALUATOR FUNCTION ---
def evaluate_model(model, test_triples_df):
    model.eval()
    results = []
    
    with torch.no_grad():
        node_emb, rel_emb = model.encode(data.edge_index, data.edge_type)
        test_heads = torch.tensor(test_triples_df['head'].values, device=device)
        test_rels = torch.tensor(test_triples_df['relation'].values, device=device)
        test_tails = torch.tensor(test_triples_df['tail'].values, device=device)
        
        for i in tqdm(range(len(test_heads)), desc="Evaluating", leave=False):
            h, r, true_t = test_heads[i], test_rels[i], test_tails[i]
            scores = model.decode(node_emb[h].unsqueeze(0), r, node_emb, rel_emb)
            sorted_indices = torch.argsort(scores, descending=True).squeeze()
            rank = (sorted_indices == true_t).nonzero(as_tuple=True)[0].item() + 1
            results.append(rank)
            
    test_triples_df['rank'] = results
    return test_triples_df

# --- 3. TRANSE MODEL CONFIGURATIONS ---
# UPDATE THESE PATHS TO YOUR KAGGLE INPUT DIRECTORY
model_configs = [
    {
        "name": "Semantic-CompGCN (TransE)",
        "class": SemanticCompGCN,
        "kwargs": {"op": "sub"},
        "path": "/kaggle/input/datasets/ullasgirish/trained-models/Semantic_CompGCN_transe.pt"
    },
    {
        "name": "Semantic-RGCN (TransE)",
        "class": SemanticRGCN,
        "kwargs": {"num_bases": 30},
        "path": "/kaggle/input/datasets/ullasgirish/trained-models/Semantic_RGCN_best_transe.pt"
    }
]

# --- 4. RUN TRANSE EVALUATION ---
all_stats = []

for config in model_configs:
    print(f"\n Loading and Testing: {config['name']}")
    model = config['class'](num_nodes, num_rels, 128, text_path, **config['kwargs']).to(device)
    model.load_state_dict(torch.load(config['path'], map_location=device))
    df_results = evaluate_model(model, df_test.copy())
    
    def get_metrics(df_subset, category_name):
        if len(df_subset) == 0: return None
        ranks = df_subset['rank'].values
        return {
            "Model": config['name'],
            "Category": category_name,
            "Test_Sample_Size": len(ranks),
            "MRR": np.mean(1.0 / ranks),
            "Hits@1(%)": np.mean(ranks <= 1) * 100,
            "Hits@3(%)": np.mean(ranks <= 3) * 100,
            "Hits@10(%)": np.mean(ranks <= 10) * 100
        }
    
    all_stats.append(get_metrics(df_results, "Global"))
    for rel_cat in ['1-to-1', '1-to-N', 'N-to-1', 'N-to-N']:
        all_stats.append(get_metrics(df_results[df_results['rel_category'] == rel_cat], rel_cat))
    all_stats.append(get_metrics(df_results[df_results['is_hub'] == True], "Hub (>=100)"))
    all_stats.append(get_metrics(df_results[df_results['is_rare'] == True], "Rare (<=10)"))

print("\n====================================================================================")
print(" TRANSE INFERENCE")
print("====================================================================================")
final_df = pd.DataFrame([s for s in all_stats if s is not None])
final_df = final_df.sort_values(by=['Category', 'MRR'], ascending=[True, False])
print(final_df.to_string(index=False, float_format="%.4f"))

### Distmult decoder

In [ ]:


distmult_configs = [
    {
        "name": "Semantic-CompGCN (DistMult)",
        "class": SemanticCompGCN,
        "kwargs": {"op": "sub"},
        "path": "/kaggle/input/datasets/ullasgirish/trained-models/Semantic_CompGCN_distmult.pt"
    },
    {
        "name": "Semantic-RGCN (DistMult)",
        "class": SemanticRGCN,
        "kwargs": {"num_bases": 30},
        "path": "/kaggle/input/datasets/ullasgirish/trained-models/Semantic_RGCN_distmult.pt"
    }
]

distmult_stats = []

for config in distmult_configs:
    print(f"\n Loading and Testing: {config['name']}")
    model = config['class'](num_nodes, num_rels, 128, text_path, **config['kwargs']).to(device)
    model.load_state_dict(torch.load(config['path'], map_location=device))
    df_results = evaluate_model(model, df_test.copy())
    
    distmult_stats.append(get_metrics(df_results, "Global"))
    for rel_cat in ['1-to-1', '1-to-N', 'N-to-1', 'N-to-N']:
        distmult_stats.append(get_metrics(df_results[df_results['rel_category'] == rel_cat], rel_cat))
    distmult_stats.append(get_metrics(df_results[df_results['is_hub'] == True], "Hub (>=100)"))
    distmult_stats.append(get_metrics(df_results[df_results['is_rare'] == True], "Rare (<=10)"))

print("\n====================================================================================")
print("DISTMULT INFERENCE ")
print("====================================================================================")
final_dm_df = pd.DataFrame([s for s in distmult_stats if s is not None])
final_dm_df = final_dm_df.sort_values(by=['Category', 'MRR'], ascending=[True, False])
print(final_dm_df.to_string(index=False, float_format="%.4f"))